# 📉➡️📈 BIST Derin Değer & Kontraryan Dip Avcısı

**Momentum kovalamayan, "aşırı ucuz kalmış ama çürük olmayan" hisseyi avlayan tarama.**

> *30 yıllık BIST tüccarı mantığı: "Malı ucuza almak kazandırır — ama sadece mal sağlamsa.
> Ucuz + çürük = değer tuzağı. Bütün maharet, ucuz-ve-sağlam ile ucuz-çünkü-batıyor'u ayırmakta."*

---

## Neden bu strateji? (Felsefe)

Bu depodaki mevcut bot (`src/scanner.py`, `src/model.py`) bir **momentum/trend-takip**
sistemidir: ADX>20, EMA hizası, yukarı kırılım, XGBoost yön tahmini. Yani **yükselişi kovalar**.

Bu notebook bunun **tam tersi** felsefeyi kurgular:

| | Momentum botu (mevcut) | Bu tarama (kontraryan derin değer) |
|---|---|---|
| Ne arar? | Yükselen, ivmeli hisse | Dövülmüş, aşırı satımda, terk edilmiş hisse |
| Ne zaman alır? | Kırılımda, kalabalıkla | Panikte, kalabalığın tersine, kademeli |
| Risk | Zirveye yakın alım | "Düşen bıçak" (kademeli alım + stop ile yönetilir) |
| Kâr mantığı | Trend devam eder | Ortalamaya dönüş + değerin fiyatı yakalaması |

**Ana belirleyici = TEKNİK göstergeler.** Bir hissenin ne kadar "aşırı ucuz" olduğunu teknik
ölçer ve sıralar. **Banker (akıllı para) + temel göstergeler ise ÜSTÜNE ekstra puanlama**
katmanıdır — çürük malı eler, sağlam olanı öne çıkarır. Alım tarafında ise **Fibonacci ile
3 kademeli alım** planı kurulur.


## 🏗️ Mimari — nasıl puanlıyoruz?

```
        ┌──────────────────── ÖNCELİKLİ ÇEKİRDEK ────────────────────┐
   OHLCV│  TEKNİK UCUZLUK (0-100)  ×0.60                              │
   ───► │   52h dip konumu·RSI(g+h)·drawdown·Bollinger%B·200EMA·W%R   │──► ÇEKİRDEK
   Hacim│                            +                                │    = sıralamayı
   ───► │  BANKER / AKILLI PARA (0-100)  ×0.40                        │      bunlar belirler
        │   CMF·MFI·A-D·OBV·pozitif diverjans (birikim)              │    + "aşırı ucuz"
        └────────────────────────────────────────────────────────────┘      KAPISI: teknik≥55
                                     ×
        ┌────────────────────────────────────────────────────────────┐
   İş Y.│  TEMEL DEĞER & KALİTE → yalnızca PUANLAMA KATKISI (±%20)     │
   / yf │   PD/DD·EV/EBITDA·F/K·DCF güvenlik marjı·owner earnings      │
   ───► │   temel_çarpan = 1 + 0.20×(temel−50)/50   (≈0.80–1.20)      │
        └────────────────────────────────────────────────────────────┘
                                     ×
        ┌────────────────────────────────────────────────────────────┐
   Temel│  VALUE-TRAP CEZASI (çarpan 0.3×–0.8× / diskalifiye)          │
   ───► │   nakit yakma·zarar·DCF pahalı·likidite·"ucuz çünkü batıyor" │
        └────────────────────────────────────────────────────────────┘
                                     ↓
   NİHAİ = ÇEKİRDEK(teknik+banker) × temel_çarpan × trap_çarpanı   (yüksek = al)
                                     ↓
              Seçilenlere → FIBONACCI 3-KADEME ALIM MERDİVENİ
```

**Öncelik sırası (kullanıcı kurgusu):** Sıralamayı **teknik ucuzluk + banker (akıllı para)**
belirler — ikisi öncelikli çekirdektir. **Temel kriterler yalnızca ölçülü bir puanlama
katkısı** (±%20) yapar; sağlam temel skoru yukarı, zayıf temel aşağı çeker ama çekirdeği
domine etmez. **Value-trap** bayrakları ise çürük malı ağır cezalar ya da tamamen eler.
Böylece en çok dövülmüş hisse otomatik "al" olmaz — hem teknik ucuz, hem akıllı para
toplamış, hem de tuzak olmayan isim öne çıkar.


## 1) Kurulum ve modüller

Bu notebook **kendine-yeterdir**: puanlama motoru (`deep_value`), veri adapteri
(`deep_value_data`), veri yükleyici ve sembol-evreni modülleri aşağıdaki hücreye **base64
ile gömülüdür**. Colab'a **yalnızca bu `.ipynb` dosyasını yükleyip çalıştırmanız yeterli** —
repo klonlamaya gerek yok. (Eğer notebook zaten repo içindeyse gömülü kopya yerine repodaki
`src/` kullanılır.) Veri kaynağı: **borsapy** (BIST'te en temiz/sağlıklı OHLCV — TradingView
backend; yfinance yalnızca yedek). İstenirse `DATA_SOURCE = "tvdatafeed"` (rongardF) da
seçilebilir. Temel oranlar: **İş Yatırım → yfinance**.

In [ ]:
# === Kurulum — KENDİNE-YETER (repo/klon gerekmez) ===
# Motor modülleri bu hücrenin içine base64 ile GÖMÜLÜdür. src/ bulunamazsa
# diske yazılır; böylece Colab'a yalnızca bu .ipynb'yi yüklemeniz yeterlidir.
import sys, subprocess, os, base64
from pathlib import Path

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# --- Bağımlılıklar ---
try:
    import pandas, numpy, matplotlib, requests, openpyxl, yfinance  # noqa
except ImportError:
    _pip("pandas", "numpy", "matplotlib", "requests", "openpyxl", "yfinance")

# --- Veri kaynağı (BIST teknik veri) ---
# VARSAYILAN "borsapy": BIST'te en TEMİZ/SAĞLIKLI veri (TradingView backend). yfinance
#   BIST'te bazı sembollerde eksik/hatalı olduğundan tercih edilmez; yalnızca yedektir.
# Seçenekler: "borsapy" | "tvdatafeed" (rongardF) | "yfinance".
#   Hangisi seçilirse seçilsin, bir sembolde başarısız olursa O SEMBOL için otomatik
#   yfinance'a düşülür; birincil kaynak sürekli çalışmıyorsa kalanlar yedeğe çevrilir.
DATA_SOURCE = "borsapy"
if DATA_SOURCE == "borsapy":
    try:
        import borsapy  # noqa
    except ImportError:
        print("borsapy kuruluyor… (bir kez)")
        _pip("borsapy")
elif DATA_SOURCE == "tvdatafeed":
    try:
        import tvDatafeed  # noqa
    except ImportError:
        print("tvdatafeed (rongardF) kuruluyor… (~1 dk, bir kez)")
        _pip("git+https://github.com/rongardF/tvdatafeed.git")
import logging as _lg
for _n in ("tvDatafeed", "tvDatafeed.main", "websocket", "borsapy"):
    _lg.getLogger(_n).setLevel(_lg.CRITICAL)   # gürültülü kaynak loglarını sustur

# --- Gömülü modül kaynakları (base64) ---
_MODS = {"config.py": "IiIiCk1lcmtlemkga29uZmlnw7xyYXN5b24gbW9kw7xsw7wuCgpCdSBtb2TDvGwsIHByb2plbmluIMOnYWzEscWfdMSxxJ/EsSBvcnRhbcSxICh5ZXJlbCBtYWtpbmUgLyBHb29nbGUgQ29sYWIgLyBHb29nbGUgRHJpdmUpCm90b21hdGlrIG9sYXJhayBhbGfEsWxhciB2ZSB0w7xtIG1vZMO8bGxlcmluIG9ydGFrIGt1bGxhbmFjYcSfxLEgeW9sLCBzYWJpdCB2ZQpwYXJhbWV0cmUgZGXEn2VybGVyaW5pIHRlayBiaXIgeWVyZGVuIHnDtm5ldGlyLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpkZWYgX2RldGVjdF9wcm9qZWN0X3Jvb3QoKSAtPiBQYXRoOgogICAgIiIiw4dhbMSxxZ9tYSBvcnRhbcSxbmEgZ8O2cmUgcHJvamUga8O2ayBkaXppbmluaSBidWx1ci4KCiAgICDDlm5jZWxpayBzxLFyYXPEsToKICAgIDEuIEJJU1RfQk9UX1JPT1Qgb3J0YW0gZGXEn2nFn2tlbmkgKGt1bGxhbsSxY8SxIGVsbGUgc2V0IGVkZXJzZSkKICAgIDIuIEdvb2dsZSBDb2xhYiArIEdvb2dsZSBEcml2ZSBtb3VudCBlZGlsbWnFn3NlIERyaXZlIGFsdMSxbmRha2kgcHJvamUga2xhc8O2csO8CiAgICAzLiBHb29nbGUgQ29sYWIgYW1hIERyaXZlIG1vdW50IGVkaWxtZW1pxZ9zZSAvY29udGVudC8gYWx0xLEKICAgIDQuIEJ1IGRvc3lhbsSxbiBidWx1bmR1xJ91IHBha2V0aW4gYmlyIMO8c3QgZGl6aW5pICh5ZXJlbCAvIEdpdEh1YiDDp2FsxLHFn21hc8SxKQogICAgIiIiCiAgICBlbnZfcm9vdCA9IG9zLmVudmlyb24uZ2V0KCJCSVNUX0JPVF9ST09UIikKICAgIGlmIGVudl9yb290OgogICAgICAgIHJldHVybiBQYXRoKGVudl9yb290KS5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCgogICAgdHJ5OgogICAgICAgIGltcG9ydCBnb29nbGUuY29sYWIgICMgbm9xYTogRjQwMQoKICAgICAgICBkcml2ZV9jYW5kaWRhdGVzID0gWwogICAgICAgICAgICBQYXRoKCIvY29udGVudC9kcml2ZS9NeURyaXZlL2Jpc3Qtc3dpbmctdHJhZGluZy1ib3QiKSwKICAgICAgICAgICAgUGF0aCgiL2NvbnRlbnQvZHJpdmUvTXkgRHJpdmUvYmlzdC1zd2luZy10cmFkaW5nLWJvdCIpLAogICAgICAgIF0KICAgICAgICBmb3IgY2FuZGlkYXRlIGluIGRyaXZlX2NhbmRpZGF0ZXM6CiAgICAgICAgICAgIGlmIGNhbmRpZGF0ZS5wYXJlbnQuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBjYW5kaWRhdGUubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgICAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZQogICAgICAgIGNvbnRlbnRfcm9vdCA9IFBhdGgoIi9jb250ZW50L2Jpc3Qtc3dpbmctdHJhZGluZy1ib3QiKQogICAgICAgIGNvbnRlbnRfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgcmV0dXJuIGNvbnRlbnRfcm9vdAogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHBhc3MKCiAgICByZXR1cm4gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKCgpQUk9KRUNUX1JPT1QgPSBfZGV0ZWN0X3Byb2plY3Rfcm9vdCgpCkNBQ0hFX0RJUiA9IFBST0pFQ1RfUk9PVCAvICJjYWNoZSIKREFUQV9DQUNIRV9ESVIgPSBDQUNIRV9ESVIgLyAiZGF0YSIKTU9ERUxfQ0FDSEVfRElSID0gQ0FDSEVfRElSIC8gIm1vZGVscyIKTE9HX0RJUiA9IFBST0pFQ1RfUk9PVCAvICJsb2dzIgoKZm9yIF9kaXIgaW4gKERBVEFfQ0FDSEVfRElSLCBNT0RFTF9DQUNIRV9ESVIsIExPR19ESVIpOgogICAgX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgojIC0tLSBWZXJpIGtheW5hxJ/EsSBheWFybGFyxLEgLS0tCkRFRkFVTFRfRVhDSEFOR0UgPSAiQklTVCIKREVGQVVMVF9JTlRFUlZBTCA9ICIxNW0iICAgICAgICAgICMgdHZkYXRhZmVlZDogaW5fMTVfbWludXRlLCB5ZmluYW5jZTogMTVtCkRFRkFVTFRfTE9PS0JBQ0tfQkFSUyA9IDUwMDAKCiMgLS0tIMOWemVsbGlrIG3DvGhlbmRpc2xpxJ9pIGF5YXJsYXLEsSAtLS0KRU1BX1dJTkRPV1MgPSAoNSwgMjEsIDUwKQpSU0lfV0lORE9XID0gMTQKQVRSX1dJTkRPVyA9IDE0CkJPTExJTkdFUl9XSU5ET1cgPSAyMApCT0xMSU5HRVJfU1REID0gMi4wCk1BQ0RfRkFTVCwgTUFDRF9TTE9XLCBNQUNEX1NJR05BTCA9IDEyLCAyNiwgOQpHQVJDSF9QLCBHQVJDSF9RID0gMSwgMQoKIyAtLS0gTW9kZWwgLyB3YWxrLWZvcndhcmQgYXlhcmxhcsSxIC0tLQpXQUxLX0ZPUldBUkRfVFJBSU5fQkFSUyA9IDEwMDAKV0FMS19GT1JXQVJEX1NURVBfQkFSUyA9IDEwMApXQUxLX0ZPUldBUkRfTUlOX1RFU1RfQkFSUyA9IDUwCkxBQkVMX0hPUklaT05fQkFSUyA9IDMgICAgICAgICAgIyBrYcOnIGJhciBpbGVyaXllIGJha2FyYWsgecO2biBldGlrZXRpIMO8cmV0aWxlY2VrCkxBQkVMX1VQX1RIUkVTSE9MRCA9IDAuMDAxNSAgICAgIyB5dWthcsSxIHnDtm4gacOnaW4gbWluaW11bSBsb2cgZ2V0aXJpIGXFn2nEn2kKClhHQl9QQVJBTV9TUEFDRSA9IHsKICAgICJuX2VzdGltYXRvcnMiOiAoMTAwLCA2MDApLAogICAgIm1heF9kZXB0aCI6ICgzLCA4KSwKICAgICJsZWFybmluZ19yYXRlIjogKDAuMDEsIDAuMyksCn0KWEdCX0RFRkFVTFRfUEFSQU1TID0gewogICAgIm5fZXN0aW1hdG9ycyI6IDMwMCwKICAgICJtYXhfZGVwdGgiOiA1LAogICAgImxlYXJuaW5nX3JhdGUiOiAwLjA1LAogICAgInN1YnNhbXBsZSI6IDAuOCwKICAgICJjb2xzYW1wbGVfYnl0cmVlIjogMC44LAogICAgIm9iamVjdGl2ZSI6ICJiaW5hcnk6bG9naXN0aWMiLAogICAgImV2YWxfbWV0cmljIjogImxvZ2xvc3MiLAogICAgIm5fam9icyI6IC0xLAp9CgojIC0tLSBSaXNrIHnDtm5ldGltaSAvIHNjYWxwaW5nIGt1cmFsbGFyxLEgLS0tCkVOVFJZX1BST0JBQklMSVRZX1RIUkVTSE9MRCA9IDAuNjUKRVhJVF9QUk9CQUJJTElUWV9USFJFU0hPTEQgPSAwLjM1ICAgIyBtb2RlbCBnw7Zyw7zFn8O8IHRlcnNpbmUgZMO2bmVyc2UgZXJrZW4gw6fEsWvEscWfCkFUUl9TVE9QX01VTFRJUExJRVIgPSAxLjUKQVRSX1RSQUlMSU5HX01VTFRJUExJRVIgPSAxLjAKQVRSX1RBS0VfUFJPRklUX01VTFRJUExJRVIgPSAyLjUKUklTS19GUkVFX1JBVEVfQU5OVUFMID0gMC4wCgojIC0tLSBQb3ppc3lvbiBib3l1dGxhbmTEsXJtYSB2ZSBkZXZyZSBrZXNpY2kgKGNpcmN1aXQgYnJlYWtlcikgLS0tClBPU0lUSU9OX1NJWkVfTU9ERSA9ICJ2b2xfdGFyZ2V0IiAgICMgImZpeGVkIiAoc2FiaXQgJSkgdmV5YSAidm9sX3RhcmdldCIgKHJpc2sgYmF6bMSxKQpSSVNLX1BFUl9UUkFERV9QQ1QgPSAwLjAxICAgICAgICAgICAjIHZvbF90YXJnZXQgbW9kdW5kYSBpxZ9sZW0gYmHFn8SxbmEgcmlza2UgZWRpbGVuIHNlcm1heWUgb3JhbsSxCk1BWF9DT05TRUNVVElWRV9MT1NTRVMgPSAzICAgICAgICAgICMgYnUgc2F5xLFkYSDDvHN0IMO8c3RlIHphcmFybMSxIGnFn2xlbWRlbiBzb25yYSBzb8SfdW1hCkNPT0xET1dOX0JBUlNfQUZURVJfTE9TU0VTID0gMTAgICAgICMgc2/En3VtYSBzw7xyZXNpIChiYXIgc2F5xLFzxLEpCk1BWF9EQUlMWV9MT1NTX1BDVCA9IDAuMDMgICAgICAgICAgICMgZ8O8bmzDvGsgemFyYXIgYnUgb3JhbsSxIGHFn2Fyc2EgbyBnw7xuIHllbmkgacWfbGVtIGHDp8SxbG1hegoKIyAtLS0gRWsgc2lueWFsIGZpbHRyZWxlcmkgKGdlcmVraXJzZSBkZXZyZSBkxLHFn8SxIGLEsXJha8SxbGFiaWxpcikgLS0tCkFEWF9XSU5ET1cgPSAxNApNSU5fQURYID0gMjAuMCAgICAgICAgICAgICAgICAgICAgICAjIGJ1IGRlxJ9lcmluIGFsdMSxbmRhICh5YXRheSBwaXlhc2EpIGnFn2xlbSBhw6fEsWxtYXoKVk9MVU1FX1dJTkRPVyA9IDIwCk1JTl9WT0xVTUVfUkFUSU8gPSAwLjcgICAgICAgICAgICAgICMgb3J0YWxhbWEgaGFjbWluIGJ1IG9yYW7EsW4gYWx0xLFuZGFraSBiYXJsYXIgZWxlbmlyClZPTF9SRUdJTUVfV0lORE9XID0gMTAwClZPTF9SRUdJTUVfTE9XRVJfUENUID0gMC4wNSAgICAgICAgICMgYcWfxLFyxLEgc2FraW4gKGTDvMWfw7xrIG95bmFrbMSxaykgcmVqaW1pIGVsZQpWT0xfUkVHSU1FX1VQUEVSX1BDVCA9IDAuOTUgICAgICAgICAjIGHFn8SxcsSxIG95bmFrIChrYXltYS9nYXAgcmlza2kgecO8a3NlaykgcmVqaW1pIGVsZQpSRVFVSVJFX1RSRU5EX0FMSUdOTUVOVCA9IFRydWUgICAgICAjIHNhZGVjZSBFTUEoZmFzdCkgPiBFTUEoc2xvdykgaWtlbiBsb25nIGHDpwpUUkVORF9GQVNUX0VNQSA9IDIxClRSRU5EX1NMT1dfRU1BID0gNTAKU0VTU0lPTl9GSUxURVJfRU5BQkxFRCA9IFRydWUKTUFSS0VUX09QRU4gPSAiMTA6MDAiICAgICAgICAgICAgICAgIyBCSVNUIHNlYW5zIGHDp8SxbMSxxZ/EsQpNQVJLRVRfQ0xPU0UgPSAiMTg6MDAiICAgICAgICAgICAgICAjIEJJU1Qgc2VhbnMga2FwYW7EscWfxLEKU0VTU0lPTl9FREdFX0VYQ0xVREVfTUlOVVRFUyA9IDE1ICAgIyBhw6fEsWzEscWfL2thcGFuxLHFn2EgeWFrxLFuIGfDvHLDvGx0w7xsw7wgZGFraWthbGFyCgojIC0tLSBQaXlhc2EgdGFyYW1hc8SxIChzY2FubmVyKSAtLS0KIyBCSVNUIDEwMCAoWFUxMDApIGVuZGVrc2kgYmlsZcWfZW5sZXJpLiBaYW1hbmxhIGVuZGVrcyBiaWxlxZ9pbWkgZGXEn2nFn2ViaWxlY2XEn2luZGVuCiMgKGdpcmVuL8OnxLFrYW4gaGlzc2VsZXIpIGJ1IGxpc3RleWkgcGVyaXlvZGlrIG9sYXJhayBnw7xuY2VsbGVtZW5peiDDtm5lcmlsaXIuCkJJU1QxMDBfU1lNQk9MUyA9IFsKICAgICJCVENJTSIsICJLVVlBUyIsICJUQ0VMTCIsICJUVEtPTSIsICJWRVNUTCIsICJQRVRLTSIsICJTSVNFIiwgIk1HUk9TIiwKICAgICJFTktBSSIsICJZS0JOSyIsICJJU01FTiIsICJIQUxLQiIsICJBS1NFTiIsICJUU0tCIiwgIkRPQVMiLCAiWk9SRU4iLAogICAgIlZBS0JOIiwgIkRPSE9MIiwgIlNLQk5LIiwgIkFLQk5LIiwgIkdTUkFZIiwgIlNBUktZIiwgIkZFTkVSIiwgIlRLRkVOIiwKICAgICJCSU1BUyIsICJCUlNBTiIsICJBTlNHUiIsICJHQVJBTiIsICJGUk9UTyIsICJUVVBSUyIsICJFQ0lMQyIsICJCU09LRSIsCiAgICAiVE9BU08iLCAiT0RBUyIsICJLUkRNRCIsICJBU0VMUyIsICJDSU1TQSIsICJFUkVHTCIsICJFS0dZTyIsICJBTEFSSyIsCiAgICAiS0NIT0wiLCAiUEdTVVMiLCAiQVJDTEsiLCAiSVNDVFIiLCAiVFVLQVMiLCAiVUxLRVIiLCAiQ0NPTEEiLCAiQlJZQVQiLAogICAgIlRIWUFPIiwgIkhFS1RTIiwgIklFWUhPIiwgIkFFRkVTIiwgIlRBVkhMIiwgIlNBU0EiLCAiT1RLQVIiLCAiU0FIT0wiLAogICAgIkFLU0EiLCAiR1VCUkYiLCAiTUFWSSIsICJCRVJBIiwgIkVOSlNBIiwgIk1QQVJLIiwgIlJBTFlIIiwgIlNPS00iLAogICAgIk9ZQUtDIiwgIlRVUlNHIiwgIkVTRU4iLCAiUVVBR1IiLCAiQ0FOVEUiLCAiR0VOSUwiLCAiR0VTQU4iLCAiTUFHRU4iLAogICAgIk1JQVRLIiwgIlBTR1lPIiwgIkRBUEdNIiwgIkdSU0VMIiwgIkVVUkVOIiwgIktMUkhPIiwgIkFTVE9SIiwgIkNWS01EIiwKICAgICJFVVBXUiIsICJDV0VORSIsICJLVExFViIsICJQQVNFVSIsICJJWkVOUiIsICJFTkVSWSIsICJSRUVEUiIsICJQQVRFSyIsCiAgICAiT0JBTVMiLCAiT0RJTkUiLCAiQUxUTlkiLCAiR1JUSE8iLCAiR0xSTUsiLCAiRFNUS0YiLCAiQkFMU1UiLCAiRUZPUiIsCiAgICAiUEFIT0wiLCAiVFJFTkoiLCAiVFJNRVQiLCAiVFJBTFQiLApdCgojIEJpciBoaXNzZSBpw6dpbiDDtm5iZWxsZWt0ZWtpIG1vZGVsIGJ1IGthZGFyIGfDvG5kZW4gZXNraXlzZSBvdG9tYXRpayBvbGFyYWsKIyB5ZW5pZGVuIGXEn2l0aWxpciAoImtlbmRpIGtlbmRpbmkgZ8O8bmNlbGxleWVuIiB0YXJhbWEgacOnaW4gdGF6ZWxpayBlxZ9pxJ9pKS4KTU9ERUxfTUFYX0FHRV9EQVlTID0gNwpTQ0FOTkVSX1RPUF9OID0gMTAKIyBTZW1ib2wgYmHFn8SxbmEgdGVrIHNlZmVybGlrICh3YWxrLWZvcndhcmQgZMO2bmfDvHPDvHopIGXEn2l0aW0gacOnaW4gcGVuY2VyZS9vcHRpbWl6YXN5b24gYXlhcmxhcsSxClNDQU5ORVJfVFJBSU5fQkFSUyA9IDc1MApTQ0FOTkVSX09QVFVOQV9UUklBTFMgPSAxNQoKIyAtLS0gQ2FubMSxIHRhcmFtYSBkw7ZuZ8O8c8O8IChzcmMvbGl2ZV9zY2FubmVyLnB5KSAtLS0KTUFSS0VUX1RJTUVaT05FID0gIkV1cm9wZS9Jc3RhbmJ1bCIKREFUQV9ERUxBWV9NSU5VVEVTID0gMTUgICAgICAgICAjIMO8Y3JldHNpeiB2ZXJpIGtheW5ha2xhcsSxbmRhIChUcmFkaW5nVmlldy95ZmluYW5jZSkgdGlwaWsgZ2VjaWttZQpMSVZFX1NDQU5fSU5URVJWQUxfTUlOVVRFUyA9IDE1ICMgdGFyYW1hIHBlcml5b2R1OyB2ZXJpIGdlY2lrbWVzaXlsZSBlxZ9sZcWfZWNlayDFn2VraWxkZSBheWFybGFubcSxxZ90xLFyCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoZGFoYSBzxLFrIHRhcmFtYW7EsW4gZmF5ZGFzxLEgeW9rdHVyIC0gYXluxLEgZ2VjaWttZWxpIHZlcmkgZMO2bmVyKQoKIyAtLS0gR2VuZWwgLS0tClJBTkRPTV9TVEFURSA9IDQyCg==", "data_loader.py": "IiIiCnNyYy9kYXRhX2xvYWRlci5weQo9PT09PT09PT09PT09PT09PT09CgrDlm5iZWxsZWsgKG1lbW9yeSArIGRpc2spIGRlc3Rla2xpLCBtaW5pbXVtIHNhecSxZGEgQVBJIGlzdGXEn2kgeWFwYW4gdmVyaSBrYXRtYW7EsS4KClRhc2FyxLFtIGlsa2VzaTogIkJpciBrZXJlIGluZGlyLCBoZXAga3VsbGFuLiIKLSDEsGxrIMOnYcSfcsSxZGEgdMO8bSBnZcOnbWnFnyB2ZXJpIChuX2JhcnMpIGluZGlyaWxpciB2ZSBoZW0gUkFNJ2RlIChwYW5kYXMgRGF0YUZyYW1lKQogIGhlbSBkZSBkaXNrdGUgKHBhcnF1ZXQpIHNha2xhbsSxci4KLSBTb25yYWtpIGhlciDDp2HEn3LEsWRhIHNhZGVjZSBzb24gYmlya2HDpyBiYXIgVHJhZGluZ1ZpZXcveWZpbmFuY2UndGFuIMOnZWtpbGlyLAogIHphdGVuIGJlbGxla3RlIG9sYW4gRGF0YUZyYW1lJ2UgImFwcGVuZCIgZWRpbGlyOyB0w7xtIGdlw6dtacWfIHRla3JhciBpbmRpcmlsbWV6LgotIEJpcmluY2lsIGtheW5hazogdHZkYXRhZmVlZCAoVHJhZGluZ1ZpZXcpLiBLaW1saWsgZG/En3J1bGFtYSBiYcWfYXLEsXPEsXogb2x1ciwKICBwYWtldCBrdXJ1bHUgZGXEn2lsc2UgdmV5YSBpc3RlayBoYXRhIHZlcmlyc2Ugb3RvbWF0aWsgb2xhcmFrIHlmaW5hbmNlJ2EgZMO8xZ9lci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBzcmMgaW1wb3J0IGNvbmZpZwoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoImJpc3RfYm90LmRhdGFfbG9hZGVyIikKaWYgbm90IGxvZ2dlci5oYW5kbGVyczoKICAgIGhhbmRsZXIgPSBsb2dnaW5nLlN0cmVhbUhhbmRsZXIoKQogICAgaGFuZGxlci5zZXRGb3JtYXR0ZXIobG9nZ2luZy5Gb3JtYXR0ZXIoIiUoYXNjdGltZSlzIFslKGxldmVsbmFtZSlzXSAlKG5hbWUpczogJShtZXNzYWdlKXMiKSkKICAgIGxvZ2dlci5hZGRIYW5kbGVyKGhhbmRsZXIpCmxvZ2dlci5zZXRMZXZlbChsb2dnaW5nLklORk8pCgpPSExDVl9DT0xVTU5TID0gWyJvcGVuIiwgImhpZ2giLCAibG93IiwgImNsb3NlIiwgInZvbHVtZSJdCgojIHR2ZGF0YWZlZWQgPC0+IHlmaW5hbmNlIDwtPiBpbnNhbi1va3VudXIgaW50ZXJ2YWwgZcWfbGVtZXNpCl9UVl9JTlRFUlZBTF9OQU1FUyA9IHsKICAgICIxbSI6ICJpbl8xX21pbnV0ZSIsCiAgICAiNW0iOiAiaW5fNV9taW51dGUiLAogICAgIjE1bSI6ICJpbl8xNV9taW51dGUiLAogICAgIjMwbSI6ICJpbl8zMF9taW51dGUiLAogICAgIjFoIjogImluXzFfaG91ciIsCiAgICAiNGgiOiAiaW5fNF9ob3VyIiwKICAgICIxZCI6ICJpbl9kYWlseSIsCn0KX1lGX0lOVEVSVkFMX01BUCA9IHsKICAgICIxbSI6ICIxbSIsCiAgICAiNW0iOiAiNW0iLAogICAgIjE1bSI6ICIxNW0iLAogICAgIjMwbSI6ICIzMG0iLAogICAgIjFoIjogIjYwbSIsCiAgICAiNGgiOiAiNjBtIiwgICMgeWZpbmFuY2UgNGggZGVzdGVrbGVtZXo7IDYwbSBpbmRpcmlsaXAgc29ucmFkYW4gcmVzYW1wbGUgZWRpbGViaWxpcgogICAgIjFkIjogIjFkIiwKfQoKCmRlZiBfYmlzdF9zeW1ib2xfZm9yX3lmaW5hbmNlKHN5bWJvbDogc3RyKSAtPiBzdHI6CiAgICAiIiJCSVNUIHNlbWJvbGxlcmluaSB5ZmluYW5jZSBmb3JtYXTEsW5hIMOnZXZpcmlyIChHQVJBTiAtPiBHQVJBTi5JUykuIiIiCiAgICBzeW1ib2wgPSBzeW1ib2wudXBwZXIoKS5zdHJpcCgpCiAgICBpZiBzeW1ib2wuZW5kc3dpdGgoIi5JUyIpOgogICAgICAgIHJldHVybiBzeW1ib2wKICAgIHJldHVybiBmIntzeW1ib2x9LklTIgoKCmRlZiBfbm9ybWFsaXplX2NvbHVtbnMoZGY6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiRmFya2zEsSBrYXluYWtsYXJkYW4gZ2VsZW4ga29sb24gaXNpbWxlcmluaSBzdGFuZGFydCBoYWxlIGdldGlyaXIuIiIiCiAgICBkZiA9IGRmLmNvcHkoKQogICAgZGYuY29sdW1ucyA9IFtzdHIoYykubG93ZXIoKSBmb3IgYyBpbiBkZi5jb2x1bW5zXQogICAgcmVuYW1lX21hcCA9IHsiYWRqIGNsb3NlIjogImNsb3NlIiwgInZvbCI6ICJ2b2x1bWUifQogICAgZGYgPSBkZi5yZW5hbWUoY29sdW1ucz1yZW5hbWVfbWFwKQogICAgbWlzc2luZyA9IFtjIGZvciBjIGluIE9ITENWX0NPTFVNTlMgaWYgYyBub3QgaW4gZGYuY29sdW1uc10KICAgIGlmIG1pc3Npbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkJla2xlbmVuIE9ITENWIGtvbG9ubGFyxLEgZWtzaWs6IHttaXNzaW5nfS4gTWV2Y3V0OiB7bGlzdChkZi5jb2x1bW5zKX0iKQogICAgZGYgPSBkZltPSExDVl9DT0xVTU5TXQogICAgZGYuaW5kZXgubmFtZSA9ICJkYXRldGltZSIKICAgIGlmIGRmLmluZGV4LnR6IGlzIG5vdCBOb25lOgogICAgICAgIGRmLmluZGV4ID0gZGYuaW5kZXgudHpfbG9jYWxpemUoTm9uZSkKICAgIHJldHVybiBkZi5zb3J0X2luZGV4KCkKCgpjbGFzcyBCaXN0RGF0YUxvYWRlcjoKICAgICIiIkJJU1QgaGlzc2VsZXJpIGnDp2luIMO2bmJlbGxla2xpIChtZW1vcnkgKyBkaXNrKSBPSExDViB2ZXJpIHnDvGtsZXlpY2kuCgogICAgUGFyYW1ldGVycwogICAgLS0tLS0tLS0tLQogICAgZXhjaGFuZ2U6IFRyYWRpbmdWaWV3IGJvcnNhIGtvZHUgKHZhcnNhecSxbGFuICJCSVNUIikuCiAgICBpbnRlcnZhbDogIjFtIiwgIjVtIiwgIjE1bSIsICIzMG0iLCAiMWgiLCAiNGgiLCAiMWQiIGRlxJ9lcmxlcmluZGVuIGJpcmkuCiAgICB0dl91c2VybmFtZSAvIHR2X3Bhc3N3b3JkOiB0dmRhdGFmZWVkIGnDp2luIG9wc2l5b25lbCBUcmFkaW5nVmlldyBraW1saWsgYmlsZ2lsZXJpLgogICAgICAgIEJlbGlydGlsbWV6c2UgdHZkYXRhZmVlZCBhbm9uaW0gKHPEsW7EsXJsxLEpIG1vZGRhIGRlbmVyOyBiYcWfYXLEsXPEsXogb2x1cnNhCiAgICAgICAgb3RvbWF0aWsgb2xhcmFrIHlmaW5hbmNlIGt1bGxhbsSxbMSxci4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGV4Y2hhbmdlOiBzdHIgPSBjb25maWcuREVGQVVMVF9FWENIQU5HRSwKICAgICAgICBpbnRlcnZhbDogc3RyID0gY29uZmlnLkRFRkFVTFRfSU5URVJWQUwsCiAgICAgICAgdHZfdXNlcm5hbWU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAogICAgICAgIHR2X3Bhc3N3b3JkOiBPcHRpb25hbFtzdHJdID0gTm9uZSwKICAgICAgICBjYWNoZV9kaXI6IE9wdGlvbmFsW1BhdGhdID0gTm9uZSwKICAgICk6CiAgICAgICAgaWYgaW50ZXJ2YWwgbm90IGluIF9UVl9JTlRFUlZBTF9OQU1FUzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkRlc3Rla2xlbm1leWVuIGludGVydmFsOiB7aW50ZXJ2YWx9LiBTZcOnZW5la2xlcjoge2xpc3QoX1RWX0lOVEVSVkFMX05BTUVTKX0iKQoKICAgICAgICBzZWxmLmV4Y2hhbmdlID0gZXhjaGFuZ2UKICAgICAgICBzZWxmLmludGVydmFsID0gaW50ZXJ2YWwKICAgICAgICBzZWxmLnR2X3VzZXJuYW1lID0gdHZfdXNlcm5hbWUKICAgICAgICBzZWxmLnR2X3Bhc3N3b3JkID0gdHZfcGFzc3dvcmQKICAgICAgICBzZWxmLmNhY2hlX2RpciA9IFBhdGgoY2FjaGVfZGlyKSBpZiBjYWNoZV9kaXIgZWxzZSBjb25maWcuREFUQV9DQUNIRV9ESVIKICAgICAgICBzZWxmLmNhY2hlX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgICAgIHNlbGYuX21lbW9yeTogZGljdFtzdHIsIHBkLkRhdGFGcmFtZV0gPSB7fQogICAgICAgIHNlbGYuX3R2X2NsaWVudCA9IE5vbmUKICAgICAgICBzZWxmLl90dl91bmF2YWlsYWJsZSA9IEZhbHNlCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwogICAgIyBLYW11eWEgYcOnxLFrIEFQSQogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwogICAgZGVmIGdldF9oaXN0b3J5KAogICAgICAgIHNlbGYsCiAgICAgICAgc3ltYm9sOiBzdHIsCiAgICAgICAgbl9iYXJzOiBpbnQgPSBjb25maWcuREVGQVVMVF9MT09LQkFDS19CQVJTLAogICAgICAgIGZvcmNlX3JlZnJlc2g6IGJvb2wgPSBGYWxzZSwKICAgICkgLT4gcGQuRGF0YUZyYW1lOgogICAgICAgICIiIlNlbWJvbCBpw6dpbiBPSExDViBnZcOnbWnFn2luaSBkw7ZuZMO8csO8ci4KCiAgICAgICAgLSBCZWxsZWt0ZSB2ZXJpIHZhcnNhOiBzYWRlY2Ugc29uIGJhcihsYXIpIMOnZWtpbGlwIG1ldmN1dCBEYXRhRnJhbWUnZSBla2xlbmlyLgogICAgICAgIC0gQmVsbGVrdGUgeW9rc2EgYW1hIGRpc2t0ZSB2YXJzYTogZGlza3RlbiB5w7xrbGVuaXIsIHNvbnJhIHNvbiBiYXIgZ8O8bmNlbGxlbmlyLgogICAgICAgIC0gSGnDp2JpcmkgeW9rc2E6IHRhbSBuX2JhcnMga2FkYXIgZ2XDp21pxZ8gaW5kaXJpbGlyLgogICAgICAgICIiIgogICAgICAgIGtleSA9IHN5bWJvbC51cHBlcigpCgogICAgICAgIGlmIGZvcmNlX3JlZnJlc2ggb3Iga2V5IG5vdCBpbiBzZWxmLl9tZW1vcnk6CiAgICAgICAgICAgIGRpc2tfZGYgPSBOb25lIGlmIGZvcmNlX3JlZnJlc2ggZWxzZSBzZWxmLl9sb2FkX2Rpc2tfY2FjaGUoa2V5KQogICAgICAgICAgICBpZiBkaXNrX2RmIGlzIG5vdCBOb25lIGFuZCBub3QgZGlza19kZi5lbXB0eToKICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdIERpc2sgw7ZuYmVsbGXEn2luZGVuICVkIGJhciB5w7xrbGVuZGkuIiwga2V5LCBsZW4oZGlza19kZikpCiAgICAgICAgICAgICAgICBzZWxmLl9tZW1vcnlba2V5XSA9IGRpc2tfZGYKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGxvZ2dlci5pbmZvKCJbJXNdIMOWbmJlbGxlayBidWx1bmFtYWTEsSwgJWQgYmFybMSxayB0YW0gZ2XDp21pxZ8gaW5kaXJpbGl5b3IuIiwga2V5LCBuX2JhcnMpCiAgICAgICAgICAgICAgICBmdWxsX2RmID0gc2VsZi5fZmV0Y2hfcmF3KGtleSwgbl9iYXJzPW5fYmFycykKICAgICAgICAgICAgICAgIHNlbGYuX21lbW9yeVtrZXldID0gZnVsbF9kZgogICAgICAgICAgICAgICAgc2VsZi5fc2F2ZV9kaXNrX2NhY2hlKGtleSwgZnVsbF9kZikKICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLl9tZW1vcnlba2V5XQoKICAgICAgICBzZWxmLl9hcHBlbmRfbGF0ZXN0X2JhcihrZXkpCiAgICAgICAgcmV0dXJuIHNlbGYuX21lbW9yeVtrZXldCgogICAgZGVmIGdldF9sYXRlc3RfYmFyKHNlbGYsIHN5bWJvbDogc3RyKSAtPiBPcHRpb25hbFtwZC5TZXJpZXNdOgogICAgICAgICIiIlNhZGVjZSBlbiBnw7xuY2VsIChzb24pIGJhcsSxIGTDtm5kw7xyw7xyLCBEYXRhRnJhbWUnaSBnw7xuY2VsbGVyLiIiIgogICAgICAgIGtleSA9IHN5bWJvbC51cHBlcigpCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWxmLl9tZW1vcnk6CiAgICAgICAgICAgIHNlbGYuZ2V0X2hpc3Rvcnkoa2V5KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuX2FwcGVuZF9sYXRlc3RfYmFyKGtleSkKICAgICAgICBpZiBzZWxmLl9tZW1vcnlba2V5XS5lbXB0eToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICByZXR1cm4gc2VsZi5fbWVtb3J5W2tleV0uaWxvY1stMV0KCiAgICBkZWYgd2FybV9jYWNoZShzZWxmLCBzeW1ib2xzOiBsaXN0W3N0cl0sIG5fYmFyczogaW50ID0gY29uZmlnLkRFRkFVTFRfTE9PS0JBQ0tfQkFSUykgLT4gTm9uZToKICAgICAgICAiIiJCaXJkZW4gw6dvayBzZW1ib2wgacOnaW4gw7ZuYmVsbGXEn2kgw7ZuY2VkZW4gxLFzxLF0xLFyICh0b3BsdSBpbGsgaW5kaXJtZSkuIiIiCiAgICAgICAgZm9yIHN5bSBpbiBzeW1ib2xzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBzZWxmLmdldF9oaXN0b3J5KHN5bSwgbl9iYXJzPW5fYmFycykKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIlslc10gw7ZuYmVsbGVrIMSxc8SxdG1hIGJhxZ9hcsSxc8SxejogJXMiLCBzeW0sIGV4YykKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiAgICAjIEJlbGxlayBnw7xuY2VsbGVtZSAoYXBwZW5kLW9ubHkpCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiAgICBkZWYgX2FwcGVuZF9sYXRlc3RfYmFyKHNlbGYsIGtleTogc3RyKSAtPiBOb25lOgogICAgICAgICIiIlNhZGVjZSBlbiBzb24gYmFybGFyxLEgw6dla2lwIG1ldmN1dCBEYXRhRnJhbWUnZSBla2xlciAodGFtIGluZGlybWUgeWFwbWF6KS4iIiIKICAgICAgICBjdXJyZW50ID0gc2VsZi5fbWVtb3J5LmdldChrZXkpCiAgICAgICAgZmV0Y2hfbiA9IDUgICMgc29uIGthcGFuYW4gYmFyxLEgZ2FyYW50aSB5YWthbGFtYWsgacOnaW4ga8O8w6fDvGsgYmlyIHBlbmNlcmUKICAgICAgICB0cnk6CiAgICAgICAgICAgIGxhdGVzdCA9IHNlbGYuX2ZldGNoX3JhdyhrZXksIG5fYmFycz1mZXRjaF9uKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIlslc10gY2FubMSxIGJhciBnw7xuY2VsbGVtZXNpIGJhxZ9hcsSxc8SxeiwgbWV2Y3V0IMO2bmJlbGxlayBrb3J1bnV5b3I6ICVzIiwga2V5LCBleGMpCiAgICAgICAgICAgIHJldHVybgoKICAgICAgICBpZiBjdXJyZW50IGlzIE5vbmUgb3IgY3VycmVudC5lbXB0eToKICAgICAgICAgICAgc2VsZi5fbWVtb3J5W2tleV0gPSBsYXRlc3QKICAgICAgICAgICAgc2VsZi5fc2F2ZV9kaXNrX2NhY2hlKGtleSwgbGF0ZXN0KQogICAgICAgICAgICByZXR1cm4KCiAgICAgICAgbGFzdF90cyA9IGN1cnJlbnQuaW5kZXhbLTFdCiAgICAgICAgbmV3X3Jvd3MgPSBsYXRlc3RbbGF0ZXN0LmluZGV4ID4gbGFzdF90c10KICAgICAgICAjIFNvbiBiYXLEsW4ga2VuZGlzaSBrYXBhbm1hZGFuIHRla3JhciBnZWxkaXlzZSAoYXluxLEgdGltZXN0YW1wKSBnw7xuY2VsbGUKICAgICAgICBvdmVybGFwcGluZyA9IGxhdGVzdFtsYXRlc3QuaW5kZXggPT0gbGFzdF90c10KCiAgICAgICAgdXBkYXRlZCA9IGN1cnJlbnQKICAgICAgICBpZiBub3Qgb3ZlcmxhcHBpbmcuZW1wdHk6CiAgICAgICAgICAgIHVwZGF0ZWQgPSB1cGRhdGVkLmNvcHkoKQogICAgICAgICAgICB1cGRhdGVkLmxvY1tsYXN0X3RzXSA9IG92ZXJsYXBwaW5nLmlsb2NbLTFdCgogICAgICAgIGlmIG5vdCBuZXdfcm93cy5lbXB0eToKICAgICAgICAgICAgdXBkYXRlZCA9IHBkLmNvbmNhdChbdXBkYXRlZCwgbmV3X3Jvd3NdKQogICAgICAgICAgICBsb2dnZXIuaW5mbygiWyVzXSAlZCB5ZW5pIGJhciBiZWxsZcSfZSBla2xlbmRpIChhcHBlbmQpLiIsIGtleSwgbGVuKG5ld19yb3dzKSkKICAgICAgICAgICAgc2VsZi5fc2F2ZV9kaXNrX2NhY2hlKGtleSwgdXBkYXRlZCkKCiAgICAgICAgc2VsZi5fbWVtb3J5W2tleV0gPSB1cGRhdGVkCgogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwogICAgIyBEaXNrIMO2bmJlbGxlxJ9pCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiAgICBkZWYgX2NhY2hlX3BhdGgoc2VsZiwga2V5OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuY2FjaGVfZGlyIC8gZiJ7a2V5fV97c2VsZi5pbnRlcnZhbH0ucGFycXVldCIKCiAgICBkZWYgX2xvYWRfZGlza19jYWNoZShzZWxmLCBrZXk6IHN0cikgLT4gT3B0aW9uYWxbcGQuRGF0YUZyYW1lXToKICAgICAgICBwYXRoID0gc2VsZi5fY2FjaGVfcGF0aChrZXkpCiAgICAgICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gcGQucmVhZF9wYXJxdWV0KHBhdGgpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBsb2dnZXIud2FybmluZygiWyVzXSBkaXNrIMO2bmJlbGxlxJ9pIG9rdW5hbWFkxLEgKCVzKSwgeWVuaWRlbiBpbmRpcmlsZWNlay4iLCBrZXksIGV4YykKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgX3NhdmVfZGlza19jYWNoZShzZWxmLCBrZXk6IHN0ciwgZGY6IHBkLkRhdGFGcmFtZSkgLT4gTm9uZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGRmLnRvX3BhcnF1ZXQoc2VsZi5fY2FjaGVfcGF0aChrZXkpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgbG9nZ2VyLndhcm5pbmcoIlslc10gZGlzayDDtm5iZWxsZcSfaW5lIHlhesSxbGFtYWTEsTogJXMiLCBrZXksIGV4YykKCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiAgICAjIFNhxJ9sYXnEsWPEsWxhcjogdHZkYXRhZmVlZCAow7ZuY2VsaWtsaSkgdmUgeWZpbmFuY2UgKHllZGVrKQogICAgIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwogICAgZGVmIF9mZXRjaF9yYXcoc2VsZiwgc3ltYm9sOiBzdHIsIG5fYmFyczogaW50KSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgaWYgbm90IHNlbGYuX3R2X3VuYXZhaWxhYmxlOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBkZiA9IHNlbGYuX2ZldGNoX3R2ZGF0YWZlZWQoc3ltYm9sLCBuX2JhcnMpCiAgICAgICAgICAgICAgICBpZiBkZiBpcyBub3QgTm9uZSBhbmQgbm90IGRmLmVtcHR5OgogICAgICAgICAgICAgICAgICAgIHJldHVybiBkZgogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgICAgICBsb2dnZXIud2FybmluZygKICAgICAgICAgICAgICAgICAgICAiWyVzXSB0dmRhdGFmZWVkIGJhxZ9hcsSxc8SxeiAoJXMpLiB5ZmluYW5jZSB5ZWRlxJ9pbmUgZ2XDp2lsaXlvci4iLCBzeW1ib2wsIGV4YwogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgc2VsZi5fdHZfdW5hdmFpbGFibGUgPSBUcnVlCgogICAgICAgIHJldHVybiBzZWxmLl9mZXRjaF95ZmluYW5jZShzeW1ib2wsIG5fYmFycykKCiAgICBkZWYgX2dldF90dl9jbGllbnQoc2VsZik6CiAgICAgICAgaWYgc2VsZi5fdHZfY2xpZW50IGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fdHZfY2xpZW50CiAgICAgICAgZnJvbSB0dkRhdGFmZWVkIGltcG9ydCBUdkRhdGFmZWVkICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICBpZiBzZWxmLnR2X3VzZXJuYW1lIGFuZCBzZWxmLnR2X3Bhc3N3b3JkOgogICAgICAgICAgICBzZWxmLl90dl9jbGllbnQgPSBUdkRhdGFmZWVkKHNlbGYudHZfdXNlcm5hbWUsIHNlbGYudHZfcGFzc3dvcmQpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5fdHZfY2xpZW50ID0gVHZEYXRhZmVlZCgpCiAgICAgICAgcmV0dXJuIHNlbGYuX3R2X2NsaWVudAoKICAgIGRlZiBfZmV0Y2hfdHZkYXRhZmVlZChzZWxmLCBzeW1ib2w6IHN0ciwgbl9iYXJzOiBpbnQpIC0+IHBkLkRhdGFGcmFtZToKICAgICAgICBmcm9tIHR2RGF0YWZlZWQgaW1wb3J0IEludGVydmFsICAjIHR5cGU6IGlnbm9yZQoKICAgICAgICBjbGllbnQgPSBzZWxmLl9nZXRfdHZfY2xpZW50KCkKICAgICAgICB0dl9pbnRlcnZhbCA9IGdldGF0dHIoSW50ZXJ2YWwsIF9UVl9JTlRFUlZBTF9OQU1FU1tzZWxmLmludGVydmFsXSkKICAgICAgICByYXcgPSBjbGllbnQuZ2V0X2hpc3QoCiAgICAgICAgICAgIHN5bWJvbD1zeW1ib2wsCiAgICAgICAgICAgIGV4Y2hhbmdlPXNlbGYuZXhjaGFuZ2UsCiAgICAgICAgICAgIGludGVydmFsPXR2X2ludGVydmFsLAogICAgICAgICAgICBuX2JhcnM9bl9iYXJzLAogICAgICAgICkKICAgICAgICBpZiByYXcgaXMgTm9uZSBvciByYXcuZW1wdHk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigidHZkYXRhZmVlZCBib8WfIHZlcmkgZMO2bmTDvHJkw7wiKQogICAgICAgIHJhdyA9IHJhdy5kcm9wKGNvbHVtbnM9WyJzeW1ib2wiXSwgZXJyb3JzPSJpZ25vcmUiKQogICAgICAgIHJldHVybiBfbm9ybWFsaXplX2NvbHVtbnMocmF3KQoKICAgIGRlZiBfZmV0Y2hfeWZpbmFuY2Uoc2VsZiwgc3ltYm9sOiBzdHIsIG5fYmFyczogaW50KSAtPiBwZC5EYXRhRnJhbWU6CiAgICAgICAgaW1wb3J0IHlmaW5hbmNlIGFzIHlmCgogICAgICAgIHlmX3N5bWJvbCA9IF9iaXN0X3N5bWJvbF9mb3JfeWZpbmFuY2Uoc3ltYm9sKSBpZiBzZWxmLmV4Y2hhbmdlLnVwcGVyKCkgPT0gIkJJU1QiIGVsc2Ugc3ltYm9sCiAgICAgICAgeWZfaW50ZXJ2YWwgPSBfWUZfSU5URVJWQUxfTUFQW3NlbGYuaW50ZXJ2YWxdCgogICAgICAgICMgeWZpbmFuY2UgZGFraWthbMSxayBiYXJsYXJkYSBnZXJpeWUgZMO2bsO8ayBzw7xyZXlpIHPEsW7EsXJsYXI7IG1ha3VsIGJpciBwZW5jZXJlIHNlw6dlbGltLgogICAgICAgIHBlcmlvZCA9IHNlbGYuX3BlcmlvZF9mb3JfeWZpbmFuY2Uobl9iYXJzLCB5Zl9pbnRlcnZhbCkKICAgICAgICByYXcgPSB5Zi5kb3dubG9hZCgKICAgICAgICAgICAgeWZfc3ltYm9sLAogICAgICAgICAgICBwZXJpb2Q9cGVyaW9kLAogICAgICAgICAgICBpbnRlcnZhbD15Zl9pbnRlcnZhbCwKICAgICAgICAgICAgcHJvZ3Jlc3M9RmFsc2UsCiAgICAgICAgICAgIGF1dG9fYWRqdXN0PUZhbHNlLAogICAgICAgICAgICBtdWx0aV9sZXZlbF9pbmRleD1GYWxzZSwKICAgICAgICApCiAgICAgICAgaWYgcmF3IGlzIE5vbmUgb3IgcmF3LmVtcHR5OgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJ5ZmluYW5jZSBib8WfIHZlcmkgZMO2bmTDvHJkw7w6IHt5Zl9zeW1ib2x9IikKICAgICAgICBkZiA9IF9ub3JtYWxpemVfY29sdW1ucyhyYXcpCiAgICAgICAgcmV0dXJuIGRmLnRhaWwobl9iYXJzKQoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfcGVyaW9kX2Zvcl95ZmluYW5jZShuX2JhcnM6IGludCwgeWZfaW50ZXJ2YWw6IHN0cikgLT4gc3RyOgogICAgICAgIGlmIHlmX2ludGVydmFsID09ICIxZCI6CiAgICAgICAgICAgIHllYXJzID0gbWF4KDEsIGludChucC5jZWlsKG5fYmFycyAvIDI1MikpICsgMSkKICAgICAgICAgICAgcmV0dXJuIGYie3llYXJzfXkiCiAgICAgICAgaW50cmFkYXlfbWF4X2RheXMgPSB7IjFtIjogNywgIjVtIjogNjAsICIxNW0iOiA2MCwgIjMwbSI6IDYwLCAiNjBtIjogNzMwfQogICAgICAgIHJldHVybiBmIntpbnRyYWRheV9tYXhfZGF5cy5nZXQoeWZfaW50ZXJ2YWwsIDYwKX1kIgo=", "bist_universe.py": "IiIiCnNyYy9iaXN0X3VuaXZlcnNlLnB5Cj09PT09PT09PT09PT09PT09PT09CgpUw7xtIEJJU1Qgc2VtYm9sIGV2cmVuaW5pIGTDtm5kw7xyZW4gaGFmaWYgeWFyZMSxbWPEsSAoeWFsbsSxemNhIGByZXF1ZXN0c2AvYHBhbmRhc2AKdmUgYGNvbmZpZ2AnZSBiYcSfbMSxZMSxcjsgYHNyYy5zY2FubmVyYCfEsW4gYcSfxLFyIE1MIGJhxJ/EsW1sxLFsxLFrbGFyxLFuxLEgw6dla21leikuCgpCdSBtb2TDvGwsIG5vdGVib29rJ2EgZ8O2bcO8bGViaWxlY2VrIGthZGFyIGJhxJ/EsW1zxLF6ZMSxci4gYHNyYy5zY2FubmVyLmZldGNoX2Jpc3Rfc3ltYm9sc2AKaWxlIGF5bsSxIGnFn2kgZ8O2csO8ciBhbWEgaXpvbGUgw6dhbMSxxZ/EsXIuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGxvZ2dpbmcKCmxvZ2dlciA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKCJiaXN0X2JvdC5iaXN0X3VuaXZlcnNlIikKaWYgbm90IGxvZ2dlci5oYW5kbGVyczoKICAgIGhhbmRsZXIgPSBsb2dnaW5nLlN0cmVhbUhhbmRsZXIoKQogICAgaGFuZGxlci5zZXRGb3JtYXR0ZXIobG9nZ2luZy5Gb3JtYXR0ZXIoIiUoYXNjdGltZSlzIFslKGxldmVsbmFtZSlzXSAlKG5hbWUpczogJShtZXNzYWdlKXMiKSkKICAgIGxvZ2dlci5hZGRIYW5kbGVyKGhhbmRsZXIpCmxvZ2dlci5zZXRMZXZlbChsb2dnaW5nLklORk8pCgoKZGVmIF9mcm9tX3RyYWRpbmd2aWV3KG1pbl9sZW46IGludCA9IDIsIG1heF9sZW46IGludCA9IDcsIHRpbWVvdXQ6IGludCA9IDIwKSAtPiBsaXN0W3N0cl06CiAgICAiIiJUcmFkaW5nVmlldyBnZW5lbCB0YXJhecSxY8SxIEFQSSdzaW5kZW4gQklTVCd0ZSBpxZ9sZW0gZ8O2cmVuIHTDvG0gaGlzc2VsZXIuIiIiCiAgICBpbXBvcnQgcmVxdWVzdHMKCiAgICB1cmwgPSAiaHR0cHM6Ly9zY2FubmVyLnRyYWRpbmd2aWV3LmNvbS90dXJrZXkvc2NhbiIKICAgIGhlYWRlcnMgPSB7CiAgICAgICAgIkNvbnRlbnQtVHlwZSI6ICJhcHBsaWNhdGlvbi9qc29uIiwKICAgICAgICAiVXNlci1BZ2VudCI6ICJNb3ppbGxhLzUuMCIsCiAgICAgICAgIk9yaWdpbiI6ICJodHRwczovL3d3dy50cmFkaW5ndmlldy5jb20iLAogICAgICAgICJSZWZlcmVyIjogImh0dHBzOi8vd3d3LnRyYWRpbmd2aWV3LmNvbS8iLAogICAgfQogICAgcGF5bG9hZCA9IHsKICAgICAgICAiZmlsdGVyIjogW10sCiAgICAgICAgIm9wdGlvbnMiOiB7ImxhbmciOiAiZW4ifSwKICAgICAgICAic3ltYm9scyI6IHsicXVlcnkiOiB7InR5cGVzIjogWyJzdG9jayJdfSwgInRpY2tlcnMiOiBbXX0sCiAgICAgICAgImNvbHVtbnMiOiBbIm5hbWUiLCAiZGVzY3JpcHRpb24iLCAidHlwZSIsICJzdWJ0eXBlIiwgImV4Y2hhbmdlIl0sCiAgICAgICAgInNvcnQiOiB7InNvcnRCeSI6ICJuYW1lIiwgInNvcnRPcmRlciI6ICJhc2MifSwKICAgICAgICAicmFuZ2UiOiBbMCwgMjAwMF0sCiAgICB9CiAgICByZXNwID0gcmVxdWVzdHMucG9zdCh1cmwsIGpzb249cGF5bG9hZCwgaGVhZGVycz1oZWFkZXJzLCB0aW1lb3V0PXRpbWVvdXQpCiAgICByZXNwLnJhaXNlX2Zvcl9zdGF0dXMoKQogICAgc3ltYm9sczogbGlzdFtzdHJdID0gW10KICAgIGZvciByb3cgaW4gcmVzcC5qc29uKCkuZ2V0KCJkYXRhIiwgW10pOgogICAgICAgIGNvbHMgPSByb3cuZ2V0KCJkIikgb3IgW10KICAgICAgICB0aWNrZXIgPSBjb2xzWzBdIGlmIGNvbHMgZWxzZSBOb25lCiAgICAgICAgaWYgdGlja2VyIGFuZCBpc2luc3RhbmNlKHRpY2tlciwgc3RyKToKICAgICAgICAgICAgY2xlYW4gPSB0aWNrZXIuc3BsaXQoIjoiKVstMV0uc3RyaXAoKS51cHBlcigpCiAgICAgICAgICAgIGlmIGNsZWFuLmlzYWxwaGEoKSBhbmQgbWluX2xlbiA8PSBsZW4oY2xlYW4pIDw9IG1heF9sZW46CiAgICAgICAgICAgICAgICBzeW1ib2xzLmFwcGVuZChjbGVhbikKICAgIHJldHVybiBsaXN0KGRpY3QuZnJvbWtleXMoc3ltYm9scykpCgoKZGVmIF9mcm9tX2JvcnNhcHkobWluX2xlbjogaW50ID0gMiwgbWF4X2xlbjogaW50ID0gNykgLT4gbGlzdFtzdHJdOgogICAgIiIiYm9yc2FweSBpbGUgdMO8bSBCSVNUIGhpc3NlIHNlbWJvbGxlcmkgKEJJU1QgVMOcTSBlbmRla3NpIC8gY29tcGFuaWVzIHRhYmxvc3UpLiIiIgogICAgaW1wb3J0IGJvcnNhcHkKCiAgICAjIDEpIEJJU1QgVMOcTSAoWFVUVU0pIGVuZGVrcyBiaWxlxZ9lbmxlcmkKICAgIGZvciBhdHRyIGluICgiY29tcG9uZW50X3N5bWJvbHMiLCAiY29tcG9uZW50cyIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgdmFsID0gZ2V0YXR0cihib3JzYXB5LkluZGV4KCJYVVRVTSIpLCBhdHRyKQogICAgICAgICAgICBzeW1zID0gbGlzdCh2YWwoKSBpZiBjYWxsYWJsZSh2YWwpIGVsc2UgdmFsKQogICAgICAgICAgICBvdXQgPSBbc3RyKHMpLnNwbGl0KCI6IilbLTFdLnN0cmlwKCkudXBwZXIoKSBmb3IgcyBpbiBzeW1zXQogICAgICAgICAgICBvdXQgPSBbcyBmb3IgcyBpbiBvdXQgaWYgcy5pc2FscGhhKCkgYW5kIG1pbl9sZW4gPD0gbGVuKHMpIDw9IG1heF9sZW5dCiAgICAgICAgICAgIGlmIGxlbihvdXQpID4gMTAwOgogICAgICAgICAgICAgICAgcmV0dXJuIGxpc3QoZGljdC5mcm9ta2V5cyhvdXQpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICBjb250aW51ZQogICAgIyAyKSBjb21wYW5pZXMoKSB0YWJsb3N1CiAgICB0cnk6CiAgICAgICAgZGYgPSBib3JzYXB5LmNvbXBhbmllcygpCiAgICAgICAgZm9yIGNvbCBpbiAoInN5bWJvbCIsICJjb2RlIiwgInRpY2tlciIsICJTeW1ib2wiLCAiQ29kZSIsICJUaWNrZXIiKToKICAgICAgICAgICAgaWYgY29sIGluIGRmLmNvbHVtbnM6CiAgICAgICAgICAgICAgICBvdXQgPSBbc3RyKHgpLnNwbGl0KCI6IilbLTFdLnN0cmlwKCkudXBwZXIoKSBmb3IgeCBpbiBkZltjb2xdLmRyb3BuYSgpXQogICAgICAgICAgICAgICAgb3V0ID0gW3MgZm9yIHMgaW4gb3V0IGlmIHMuaXNhbHBoYSgpIGFuZCBtaW5fbGVuIDw9IGxlbihzKSA8PSBtYXhfbGVuXQogICAgICAgICAgICAgICAgaWYgbGVuKG91dCkgPiAxMDA6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGxpc3QoZGljdC5mcm9ta2V5cyhvdXQpKQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgcGFzcwogICAgcmV0dXJuIFtdCgoKZGVmIGZldGNoX2Jpc3Rfc3ltYm9scygpIC0+IGxpc3Rbc3RyXToKICAgICIiIkJJU1QndGUgacWfbGVtIGfDtnJlbiBUw5xNIGhpc3NlbGVyaW4gKH42MDArKSBnw7xuY2VsIGxpc3Rlc2kuCgogICAgMSkgYm9yc2FweSAoQklTVCBUw5xNKSDihpIgMikgVHJhZGluZ1ZpZXcgU2Nhbm5lciBBUEkg4oaSCiAgICAzKSBgY29uZmlnLkJJU1QxMDBfU1lNQk9MU2Agw6dla2lyZGVrIHllZGXEn2kuCiAgICAiIiIKICAgIHRyeToKICAgICAgICBzeW1ib2xzID0gX2Zyb21fYm9yc2FweSgpCiAgICAgICAgaWYgc3ltYm9sczoKICAgICAgICAgICAgbG9nZ2VyLmluZm8oImJvcnNhcHk6ICVkIEJJU1QgaGlzc2VzaSDDp2VraWxkaS4iLCBsZW4oc3ltYm9scykpCiAgICAgICAgICAgIHJldHVybiBzeW1ib2xzCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgbG9nZ2VyLndhcm5pbmcoImJvcnNhcHkgc2VtYm9sIGxpc3Rlc2kgYmHFn2FyxLFzxLF6OiAlcyIsIGV4YykKCiAgICB0cnk6CiAgICAgICAgc3ltYm9scyA9IF9mcm9tX3RyYWRpbmd2aWV3KCkKICAgICAgICBpZiBzeW1ib2xzOgogICAgICAgICAgICBsb2dnZXIuaW5mbygiVHJhZGluZ1ZpZXcgU2Nhbm5lcjogJWQgQklTVCBoaXNzZXNpIMOnZWtpbGRpLiIsIGxlbihzeW1ib2xzKSkKICAgICAgICAgICAgcmV0dXJuIHN5bWJvbHMKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBsb2dnZXIud2FybmluZygiVHJhZGluZ1ZpZXcgU2Nhbm5lciBBUEkgYmHFn2FyxLFzxLF6OiAlcyIsIGV4YykKCiAgICB0cnk6CiAgICAgICAgZnJvbSBzcmMgaW1wb3J0IGNvbmZpZwogICAgICAgIGNvcmUgPSBsaXN0KGNvbmZpZy5CSVNUMTAwX1NZTUJPTFMpCiAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBjb3JlID0gW10KICAgIGxvZ2dlci53YXJuaW5nKCJEaW5hbWlrIGtheW5hayBiYcWfYXLEsXPEsXo7IMOnZWtpcmRlayBsaXN0ZSBrdWxsYW7EsWzEsXlvciAoJWQgaGlzc2UpLiIsIGxlbihjb3JlKSkKICAgIHJldHVybiBjb3JlCg==", "deep_value.py": "...", "deep_value_data.py": "..."}

# Repo kökünü (varsa) bul, yoksa çalışma dizinini kullan
ROOT = os.getcwd()
for cand in [os.getcwd(), *map(str, Path(os.getcwd()).parents)]:
    if (Path(cand) / "src").is_dir():
        ROOT = cand; break

# Gömülü motor modüllerini DAİMA (üzerine) yaz — böylece oturumda kalan ESKİ/EKSİK
# bir src/ (ör. bist_universe.py'siz) sürüm/uyuşmazlık hatası çıkaramaz.
srcdir = Path(ROOT) / "src"
srcdir.mkdir(parents=True, exist_ok=True)
if not (srcdir / "__init__.py").exists():
    (srcdir / "__init__.py").write_text("")
for name, b64 in _MODS.items():
    (srcdir / name).write_bytes(base64.b64decode(b64))

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Kernelde önbelleğe alınmış eski src.* modüllerini temizle (yeniden çalıştırma güvenliği)
for _m in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

from src import deep_value as dv
from src import deep_value_data as dvd
print("Motor modülleri hazır:", str(srcdir))
print("Motor yüklendi. TECH_GATE =", dv.TECH_GATE,
      "| çekirdek ağırlıkları teknik/banker =", dv.CORE_W_TECH, "/", dv.CORE_W_BANKER,
      "| temel katkı ±", int(dv.FUND_BONUS_STRENGTH*100), "%")

## 2) Teknik Ucuzluk — öncelikli çekirdeğin 1. bileşeni (×0.60, 0-100)

100 = maksimum dövülmüş / aşırı satımda. Trend-takibin **tersine**: düşük RSI, dipteki konum,
derin drawdown **yüksek** puan alır. Alt bileşenler ve ağırlıkları:

| Bileşen | Ağırlık | Mantık |
|---|---|---|
| **52-hafta banttaki konum** | %28 | Dibe yakınlık — "aşırı ucuz"un kalbi. `(fiyat−52hDip)/(52hTepe−52hDip)` |
| **Günlük RSI(14)** | %20 | Aşırı satım (RSI 50→0 puan, ≤15→100) |
| **Haftalık RSI(14)** | %12 | Kalıcı ucuzluk teyidi (gürültüye dayanıklı) |
| **52-hafta zirveden düşüş** | %15 | Ne kadar dövülmüş (%10→0, %60→100) |
| **Bollinger %B** | %10 | Alt banda/altına sarkma |
| **200-EMA sapması** | %10 | Uzun vade ortalamanın ne kadar altında |
| **Williams %R** | %5 | Ek aşırı-satım teyidi |

Ayrıca **dip teyidi**: MACD histogramı son barlarda yukarı dönüyorsa (düşüş ivmesi kırıldı)
`dip_confirm=True`. Kapı: `teknik ≥ 55` → "aşırı ucuz aday".


## 3) BANKER / Akıllı Para — öncelikli çekirdeğin 2. bileşeni (×0.40)

**Teknik ile birlikte önceliklidir**: sıralamayı bu ikisi belirler. Aşırı ucuz bir hissede
asıl aradığımız, fiyat dibe yatarken **güçlü ellerin sessizce topladığının** izidir. Klasik
dip-avı sinyali **pozitif diverjans**tır: fiyat düşük/yatay ama para akışı yukarı.

- **CMF (Chaikin Money Flow)** > 0 → birikim (banker topluyor)
- **MFI (Money Flow Index)** → hacim ağırlıklı aşırı-satım/dönüş
- **A/D çizgisi & OBV eğimi** → kümülatif hacim baskısı yönü
- **Pozitif diverjans bonusu** → fiyat son 20 günde düşmüş **ama** A/D/CMF yukarı

Banker skoru **düşükse** hisse hâlâ *dağıtım* (satış) altındadır → "ucuz ama akıllı para
girmemiş, acele etme / daha derin kademeleri bekle". Yüksekse teknik ucuzluk daha güvenilir.


## 4) Temel Değer & Kalite — yalnızca puanlama KATKISI (±%20) + Value-Trap elemesi

Temel kriterler çekirdeği (teknik+banker) **domine etmez**; sadece ±%20'lik bir çarpanla
ince ayar yapar (`temel_çarpan = 1 + 0.20×(temel−50)/50`). İki alt blok **geometrik**
birleştirilir (ikisi de gerekli):

- **Ucuzluk (value):** PD/DD, EV/EBITDA, F/K, EV/Satış — sektör-göreli ve mutlak.
  *Eksiğe dayanıklı*: BIST'te İş Yatırım bazı oranları vermez, Yahoo trailing F/K çevrimsel
  diplerde 500+ olabilir → eldeki metriklerin ağırlıklı ortalaması alınır, tek metrik skoru
  sıfırlamaz.
- **Kalite (quality):** owner earnings (gerçek nakit üretimi), DCF güvenlik marjı, Buffett
  skoru, borçluluk, ROE.

**Value-trap bayrakları** (30 yıllık trader refleksi) nihai skoru çarpanla cezalandırır:

| Bayrak | Ceza | Örnek (2026-07-03) |
|---|---|---|
| Nakit yakıyor (negatif owner earnings) | ×0.30 | ECILC (OE −1.305M) |
| Zarar (negatif net kâr) | ×0.55 | ECILC (−924M) |
| DCF'e göre pahalı + AVOID | ×0.55 | EREGL (marj −%1175), ARCLK (−%127) |
| Ucuz çünkü zarar (F/K yok + düşük PD/DD + nakit yok) | ×0.50 | CANTE (PD/DD 0.4, OE=NaN) |
| İşletme değeri pahalı (EV/EBITDA>15) | ×0.75 | SASA (EV/EBITDA 26) |
| Likidite tuzağı / düşen bıçak | ×0.60 / ×0.80 | — |

Nakit yakan **+** zarar eden **+** AVOID kombinasyonu → **tam diskalifiye**.


## 5) Alım tarafı — Fibonacci 3-Kademeli Merdiven

"Düşen bıçağı" tek hamlede tutmayız. Seçilen her hisse için son ~180 barın baskın swing'i
(tepe **H**, dip **L**) alınır; H→L düşüşünün Fibonacci seviyeleri referanstır. Alım
kademeleri **yalnızca güncel fiyatın altındaki** seviyelere konur ve **derine daha çok
ağırlık** verilir (ucuzsa daha çok al):

- **1. kademe (%25)** — güncel fiyat / ilk Fib desteği (0.618–1.0)
- **2. kademe (%35)** — 1.272 uzantısı (kapitülasyon)
- **3. kademe (%40)** — 1.618 uzantısı (aşırı kapitülasyon)
- **HARD STOP** = en derin kademe − 1.5×ATR (bu da tutmazsa tez yanlış)

DCF içsel değeri varsa merdivenin ağırlıklı ortalama maliyetine göre **beklenen getiri**
hesaplanır (teknik giriş ↔ temel hedef köprüsü).


## 6) Universe ve tarama — **TÜM BIST**

Evren, `src.scanner.fetch_bist_symbols()` ile **TradingView Scanner API'sinden tüm BIST
hisseleri** (canlı, ~500+ sembol) olarak çekilir; API'ye ulaşılamazsa HTML tablo yedeklerine,
en sonda `config.BIST100_SYMBOLS` çekirdek listesine düşer.

**Ana belirleyici teknik olduğu için** önce her hissenin teknik ucuzluğu hesaplanır; kapıyı
(TECH_GATE) geçemeyecek kadar pahalı olanlara **temel/banker verisi çekilmez** — yüzlerce
hissede gereksiz API isteği yapılmaz, tarama makul sürede biter. Yine de tüm evren teknik
olarak taranır.

In [ ]:
from src import config
from src.bist_universe import fetch_bist_symbols
import time

# TÜM BIST evreni (canlı; başarısızsa config.BIST100_SYMBOLS çekirdek yedeğine düşer)
SYMBOLS = fetch_bist_symbols()

# HIZLI TEST: kaç hisse taransın? (güvenli ilk çalıştırma için varsayılan 30)
#   → TÜM BIST için: HIZLI_TEST = 0  (~600 hisse, birkaç dakika; çıktı bastırılır,
#     sekme donmaz, ilerleme 25'te bir yazılır). Önce 30 ile akışın çalıştığını görün.
HIZLI_TEST = 30
if HIZLI_TEST:
    SYMBOLS = SYMBOLS[:HIZLI_TEST]
print(f"Evren: {len(SYMBOLS)} BIST hissesi | veri kaynağı: {DATA_SOURCE}")

t0 = time.time()
report, results = dvd.screen_universe(
    SYMBOLS,
    n_bars=600,            # ~2.5 yıl günlük bar (200EMA + 52h için yeterli)
    fib_lookback=180,      # Fibonacci swing penceresi
    tech_prefilter=True,   # teknik kapıyı geçemeyene temel veri çekme (tüm BIST için şart)
    with_ladder_top_n=20,  # en iyi 20 adaya Fib merdiveni hesapla
    source=DATA_SOURCE,    # ilk hücredeki seçim (varsayılan yfinance)
)
print(f"\n{len(results)} hisse tarandı, {int(report['asiri_ucuz'].sum())} tanesi teknik "
      f"kapıyı geçti. ({time.time()-t0:.0f} sn)\n")
report.head(30)

### 6a) Sadece "aşırı ucuz" adaylar (kapıyı geçenler, tuzaksızlar önde)

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 30)

adaylar = report[report["asiri_ucuz"] & (~report["disqualified"])].copy()
cols = ["symbol","sector","final_score","cekirdek","tech_ucuzluk","banker","temel_skor",
        "temel_carpan","value","quality","rsi_d","rsi_w","pos_52w","drawdown","birikim","trap_mult","trap_flags"]
adaylar[cols].reset_index(drop=True)

### 6b) Değer tuzağı olarak elenenler (ders niteliğinde)

Bunlar teknik olarak ucuz **ama** temelde çürük — sistem bilerek geri iter. En çok dövülmüş
hissenin en iyi alım OLMADIĞINI gösterir.

In [ ]:
tuzaklar = report[(report["trap_flags"] != "") | (report["disqualified"])].copy()
tuzaklar[["symbol","final_score","cekirdek","tech_ucuzluk","banker","temel_skor","value","quality",
          "trap_mult","disqualified","trap_flags"]].reset_index(drop=True)

## 7) Seçilen hisse için Fibonacci 3-kademe alım grafiği

In [ ]:
import matplotlib.pyplot as plt

def plot_fib_ladder(symbol, results, n_bars=250):
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: sonuç/merdiven yok."); return
    df = dvd.load_daily_ohlcv(symbol, n_bars=600).tail(n_bars)
    L = res.ladder
    fig, ax = plt.subplots(figsize=(13, 6))
    ax.plot(df.index, df["close"], color="#1f2d3d", lw=1.3, label="Kapanış")
    ax.axhline(L.swing_high, color="#888", ls="--", lw=0.8)
    ax.text(df.index[0], L.swing_high, f"  swing tepe {L.swing_high}", va="bottom", color="#888", fontsize=8)
    palette = ["#2e7d32", "#f9a825", "#c62828"]
    for i, rung in enumerate(L.rungs):
        c = palette[min(i, 2)]
        ax.axhline(rung["price"], color=c, lw=1.4, alpha=0.9)
        ax.text(df.index[-1], rung["price"],
                f"  {i+1}. kademe %{rung['weight_pct']} @ {rung['price']}",
                va="center", color=c, fontsize=9, fontweight="bold")
    ax.axhline(L.hard_stop, color="#6a1b9a", ls=":", lw=1.2)
    ax.text(df.index[-1], L.hard_stop, f"  STOP {L.hard_stop}", va="center", color="#6a1b9a", fontsize=8)
    title = (f"{symbol} — Nihai {res.final_score} | Teknik {res.technical.total} | "
             f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if L.expected_upside_pct is not None:
        title += f"\nDCF hedef {L.dcf_target} → merdiven ort. maliyetine göre bek. getiri %{L.expected_upside_pct}"
    ax.set_title(title, fontsize=11)
    ax.legend(loc="upper right"); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

# En iyi aday için çiz:
en_iyi = report[report["asiri_ucuz"] & (~report["disqualified"])]
if not en_iyi.empty:
    plot_fib_ladder(en_iyi.iloc[0]["symbol"], results)

In [ ]:
def alim_plani(symbol, results, sermaye=100_000):
    """Bir hisse için kademeli alım planını TL bazında yazdırır."""
    res = next((r for r in results if r.symbol == symbol), None)
    if res is None or res.ladder is None:
        print(f"{symbol}: merdiven yok."); return
    L = res.ladder
    print(f"=== {symbol} — Kademeli Alım Planı (sermaye {sermaye:,.0f} TL) ===")
    print(f"Nihai skor {res.final_score} | Teknik {res.technical.total} | "
          f"Banker {res.banker.total} | Temel {res.fundamental.total}")
    if res.trap.flags:
        print("⚠️  Tuzak bayrakları:", "; ".join(res.trap.flags))
    print(f"Güncel {L.current_price} | swing {L.swing_low}–{L.swing_high} | STOP {L.hard_stop}\n")
    toplam_lot = 0
    for i, r in enumerate(L.rungs, 1):
        pay = sermaye * r["weight_pct"] / 100
        lot = int(pay / r["price"])
        toplam_lot += lot
        print(f"  {i}. kademe @ {r['price']:>8}  (%{r['weight_pct']:<4} = {pay:>10,.0f} TL "
              f"→ ~{lot:,} lot)  [{r['note']}]")
    print(f"\n  Toplam ~{toplam_lot:,} lot | STOP {L.hard_stop} "
          f"(en derin kademeden ~%{(1-L.hard_stop/L.rungs[-1]['price'])*100:.1f} aşağıda)")
    if L.expected_upside_pct is not None:
        print(f"  DCF hedef {L.dcf_target} → beklenen getiri ~%{L.expected_upside_pct}")

if not en_iyi.empty:
    alim_plani(en_iyi.iloc[0]["symbol"], results)

## 8) 📊 Detaylı Excel çıktısı

Tüm taramayı çok-sayfalı, biçimlendirilmiş bir Excel dosyasına aktarır:

| Sayfa | İçerik |
|---|---|
| **Ozet** | Sıralı özet tablo (tüm evren) |
| **Adaylar** | Teknik kapıyı geçen, tuzaksız hisseler — tüm alt bileşenlerle |
| **Tuzaklar** | Value-trap bayraklı / diskalifiye edilenler (neden elendiği) |
| **Detay** | Her hisse için teknik+banker+temel **ham değerler** (RSI, CMF, MFI, 52h, PD/DD…) |
| **Alim_Plani** | Fibonacci 3-kademe merdivenleri (fiyat, ağırlık, TL, ~lot, stop, DCF hedef) |

`final_score` kolonu renk skalasıyla (kırmızı→yeşil) boyanır.

In [ ]:
SERMAYE = 100_000   # Alım planı TL/lot dağılımı için varsayılan sermaye

xlsx_path = dvd.export_to_excel(report, results,
                                path="bist_derin_deger_tarama.xlsx",
                                capital=SERMAYE)
print("Excel oluşturuldu:", xlsx_path)

# Colab'da otomatik indir:
try:
    from google.colab import files
    files.download(xlsx_path)
except Exception:
    print("Yerel çalışma: dosya çalışma dizininde ->", xlsx_path)

## 9) 📸 Canlı örnek çıktı — 2026-07-03 (gerçek veri)

Aşağıdaki değerler, motorun **gerçek İş Yatırım temel verisi + gerçek OHLCV** üzerinde
o günkü çıktısıdır. Notebook'u çalıştırınca güncel tarih için yeniden hesaplanır.

### ULKER — tam gerçek çalıştırma (öncelikli çekirdek örneği)

| Bileşen | Değer | Not |
|---|---|---|
| **Nihai skor** | **58.9** | çekirdek × temel_çarpan × trap |
| Çekirdek (teknik+banker) | 50.8 | `0.60×67.8 + 0.40×25.4` |
| ⤷ Teknik ucuzluk | **67.8** | RSI 30.3, 52h dibinde (pos 0.08), drawdown %29, MACD dip teyidi ✅ |
| ⤷ Banker/akıllı para | **25.4** | CMF −0.12 → **hâlâ dağıtım, akıllı para girmemiş** |
| Temel katkı | ×1.158 | temel 89.5 (value 87 / quality 91) → +%16 katkı |
| Value-trap | ×1.00 | temiz |

> **Öncelik mantığı iş başında:** ULKER temelde mükemmel (89.5) olmasına rağmen **banker
> skoru düşük** olduğu için çekirdek 50.8'e iniyor — sistem "ucuz ve sağlam, ama akıllı para
> daha toplamıyor, acele etme / kademeli gir" diyor. Temel tek başına sıralamayı yukarı
> taşıyamıyor; öncelik teknik+banker'da.

### Aşırı-ucuz aday havuzu (kapıyı geçenler) — gerçek temel + RSI

Nihai sıralama canlı çalıştırmada her hissenin **banker** skoruyla belirlenir; aşağıdaki
temel skorlar (İş Yatırım) ve RSI (o gün) gerçektir:

| Hisse | RSI | Temel | value/quality | Neden aday |
|---|---|---|---|---|
| **ULKER** | 30.3 | 89.5 | 87/91 | 52h dibinde, DCF marjı %68, EV/EBITDA 4.3 |
| **PETKM** | 34.0 | 83.5 | 93/76 | PD/DD 0.7, güvenlik marjı %34, nakit üretiyor |
| **TURSG** | 37.8 | 76.3 | 55/100 | STRONG_BUY, F/K 5.6, güvenlik marjı %64 |
| **MAVI** | 38.9 | 76.7 | 68/84 | EV/EBITDA 2.9, güvenlik marjı %42 |
| **FROTO** | 36.3 | 64.9 | 65/65 | Güçlü hendek, F/K 8.5, ama marj dar %17 |

### Değer tuzağı olarak elenenler (teknik ucuz ama çürük)

| Hisse | RSI | Neden ELENDİ | Çarpan |
|---|---|---|---|
| **CANTE** | **27.4** (en oversold!) | Owner earnings YOK, PD/DD 0.4 → "ucuz çünkü zarar" | ×0.50 |
| **ECILC** | 39.7 | Nakit yakıyor (−1.305M) + zarar (−924M) + AVOID | **diskalifiye** |
| **EREGL** | 30.6 | DCF marjı −%1175 (kâr çökmüş, çevrimsel), AVOID | ×0.55 |
| **ARCLK** | – | DCF marjı −%127, düşük nakit getirisi, AVOID | ×0.55 |

> **Kilit gözlem:** En çok dövülmüş hisse (CANTE, RSI 27) **en iyi alım değil** — sistem onu
> tuzak olarak eler. Asıl alınacak, hem aşırı ucuz **hem** nakit üreten ULKER/PETKM/TURSG.

### ULKER — gerçek 3-kademe Fibonacci merdiveni (güncel 100.1 ₺)

| Kademe | Fiyat | Ağırlık | Fib | Not |
|---|---|---|---|---|
| 1 | **96.5** | %25 | 1.000 | swing dip retesti |
| 2 | **84.2** | %35 | 1.272 | kapitülasyon uzantısı |
| 3 | **68.6** | %40 | 1.618 | aşırı kapitülasyon |
| **STOP** | **64.3** | — | — | en derin kademe − 1.5×ATR |

*52h yüksek 141.7 / düşük 96.5, drawdown %29, MACD dip teyidi ✅. Banker skoru düşük olduğu
için "kademeli gir, tek seferde yükleme" mesajı verir.*


## 10) ⚠️ Risk yönetimi ve kullanım notları

1. **Kademeli gir, aşkla değil planla.** 3 kademe = maliyet ortalaması + yanlışsa kontrollü
   zarar. Banker skoru düşükken (akıllı para girmemişken) ilk kademeyi küçük tut, derin
   kademeleri bekle.
2. **STOP'a sadık kal.** En derin kademe − 1.5×ATR kırılırsa tez yanlıştır; değer tuzağına
   dönmüş olabilir. Zararı büyütme.
3. **Value-trap bayraklarını ciddiye al.** "Ucuz" bir hisse aylarca daha ucuzlayabilir.
   Nakit yakan / zarar eden / DCF'e göre pahalı isimlerden uzak dur (sistem zaten cezalandırır).
4. **Katalizör/likidite.** Derin değer, katalizör olmadan uzun sürebilir. Likidite tuzağı
   bayraklı (düşük TL hacim) isimlerde pozisyon küçük olsun.
5. **Temel veri gecikmeli/eksiktir.** İş Yatırım oranları çeyrekliktir; teyit için son bilanço
   ve KAP'a bak. F/K çevrimsel diplerde yanıltır → PD/DD ve EV/EBITDA'ya ağırlık ver.
6. **Bu bir yatırım tavsiyesi değildir.** Eğitim ve araştırma amaçlı bir tarama aracıdır.
   Kararı kendi analizinle ver.

---

### Parametre ayarları (`src/deep_value.py`)
`TECH_GATE` (aşırı-ucuz eşiği, vars. 55) · `CORE_W_TECH`/`CORE_W_BANKER` (öncelikli çekirdek
ağırlıkları, vars. 0.60/0.40) · `FUND_BONUS_STRENGTH` (temel kriter katkı gücü, ±%20) ·
teknik alt-ağırlıklar `TECH_WEIGHTS` · Fibonacci `_FIB_RATIOS` ve kademe ağırlıkları.
Banker'a daha çok öncelik vermek için `CORE_W_BANKER`'ı artır. Kendi tarzına göre kalibre et.
